# Notebook 1: Data Cleaning & ETL Pipeline
**Project:** Customer Churn Intelligence System — Telecom  
**Author:** Prathmesh Joshi  
**Dataset:** IBM Telco Customer Churn (Kaggle)

---
### Goal
Load the raw dataset, clean it, fix data type issues, handle missing values, and export a clean version ready for analysis and MySQL loading.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## Step 2: Load the Dataset

In [ ]:
# Load raw data
# Make sure you have placed the CSV in the /data folder
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

## Step 3: Initial Inspection

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

In [ ]:
# Summary statistics
df.describe()

## Step 4: Fix the TotalCharges Column

**Problem:** `TotalCharges` is stored as a string (object) instead of a number.  
Some rows have whitespace instead of a value — these are new customers with 0 tenure.

In [ ]:
# See the problem
print('TotalCharges dtype before fix:', df['TotalCharges'].dtype)
print('Sample values with spaces:', df[df['TotalCharges'].str.strip() == '']['TotalCharges'].count(), 'rows')

In [ ]:
# Replace whitespace with NaN, then convert to float
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill NaN in TotalCharges with 0 (these are new customers with no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('TotalCharges dtype after fix:', df['TotalCharges'].dtype)
print('Missing values remaining:', df['TotalCharges'].isnull().sum())

## Step 5: Encode Binary Columns

Convert Yes/No text columns to 1/0 numbers — needed for the prediction model later.

In [ ]:
# Encode the target column first
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Encode other Yes/No columns
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in yes_no_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0})

# Encode gender
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})

print('Binary encoding complete.')
df[['Churn', 'gender', 'Partner', 'PhoneService']].head()

## Step 6: Create Tenure Buckets

Group customers into tenure segments for cohort analysis.

In [ ]:
def tenure_bucket(months):
    if months <= 12:
        return '0-12 months'
    elif months <= 24:
        return '13-24 months'
    elif months <= 48:
        return '25-48 months'
    else:
        return '49+ months'

df['tenure_bucket'] = df['tenure'].apply(tenure_bucket)

print('Tenure bucket distribution:')
print(df['tenure_bucket'].value_counts())

## Step 7: Final Data Quality Check

In [ ]:
print('=== FINAL DATA QUALITY REPORT ===')
print(f'Total rows: {len(df):,}')
print(f'Total columns: {df.shape[1]}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Churn rate: {df["Churn"].mean():.1%}')
print(f'Non-churned: {(df["Churn"]==0).sum():,} customers')
print(f'Churned: {(df["Churn"]==1).sum():,} customers')

## Step 8: Export Clean Data

In [ ]:
# Save clean CSV (used by all other notebooks and Power BI)
df.to_csv('../data/telco_churn_clean.csv', index=False)
print('Clean data saved to: data/telco_churn_clean.csv')
print(f'Shape: {df.shape}')

## Step 9 (Optional): Load to MySQL

If you have MySQL installed, run this cell to load the clean data into a database.  
**Update the connection string with your own MySQL username and password.**

In [ ]:
# OPTIONAL — requires MySQL to be running
# pip install sqlalchemy pymysql

from sqlalchemy import create_engine

# UPDATE THESE WITH YOUR CREDENTIALS
username = 'root'
password = 'your_password'    # <-- change this
database = 'telecom_churn'    # <-- create this database in MySQL first
host = 'localhost'

engine = create_engine(f'mysql+pymysql://{username}:{password}@{host}/{database}')

df.to_sql('customers', con=engine, if_exists='replace', index=False)
print('Data loaded to MySQL table: customers')

---
## ✅ Notebook 1 Complete

**What we did:**
- Loaded 7,043 raw customer records
- Fixed TotalCharges data type issue (string → float)
- Encoded 11 binary columns (Yes/No → 1/0)
- Created tenure segments for cohort analysis
- Confirmed 0 missing values, 0 duplicates
- Exported clean data to `data/telco_churn_clean.csv`

**Next:** Open `02_exploratory_analysis.ipynb`